# Variational Inference in PyMC

## Learning Objectives
By completing this notebook, you will:
1. Understand variational inference as optimization-based approximate inference
2. Implement ADVI (Automatic Differentiation Variational Inference) in PyMC
3. Evaluate approximate posteriors using ELBO and diagnostic metrics
4. Apply VI to large-scale commodity forecasting problems
5. Compare VI trade-offs: speed vs accuracy

## Prerequisites
- Understanding of MCMC and NUTS (Module 6.1-6.3)
- Basic optimization concepts
- PyMC model building

## Estimated Time: 55 minutes

---

In [ ]:
learning_objectives(["Understand variational inference as optimization-based approximate inference", "Implement ADVI (Automatic Differentiation Variational Inference) in PyMC", "Evaluate approximate posteriors using ELBO and diagnostic metrics", "Apply VI to large-scale commodity forecasting problems", "Compare VI trade-offs: speed vs accuracy", "Understanding of MCMC and NUTS (Module 6.1-6.3)"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

## 1. Setup and Imports

In [ ]:
# Purpose: Import libraries for variational inference
# Key Concept: VI turns inference into an optimization problem

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import arviz as az
from scipy import stats
import time
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)
az.style.use('arviz-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"PyMC version: {pm.__version__}")
print(f"ArviZ version: {az.__version__}")

## 2. Variational Inference: Core Concepts

### The Idea

Instead of sampling from the posterior $p(\theta|y)$, we:
1. Choose a family of distributions $q(\theta; \phi)$ parameterized by $\phi$
2. Find $\phi^*$ that makes $q$ closest to $p$ (minimizes KL divergence)
3. Use $q(\theta; \phi^*)$ as an approximation to $p(\theta|y)$

### ELBO (Evidence Lower Bound)

We maximize:
$$\text{ELBO}(\phi) = \mathbb{E}_{q(\theta;\phi)}[\log p(y, \theta)] - \mathbb{E}_{q(\theta;\phi)}[\log q(\theta;\phi)]$$

### Advantages
- **Fast**: Optimization vs sampling
- **Scalable**: Works with large datasets
- **Deterministic**: Same result each run

### Disadvantages
- **Approximate**: Underestimates uncertainty
- **Mean-field assumption**: Often assumes independence
- **Local optima**: May not find global optimum

## 3. ADVI in PyMC: Basic Example

Let's start with a simple linear regression and compare NUTS vs ADVI.

In [ ]:
# Purpose: Compare NUTS and ADVI on simple regression
# Key Concept: ADVI is much faster but approximate

# Generate synthetic commodity price data
np.random.seed(123)
n = 500
X = np.random.randn(n, 3)  # 3 predictors (e.g., supply, demand, macro)
true_beta = np.array([2.0, -1.5, 0.8])
true_sigma = 1.0
y = X @ true_beta + np.random.randn(n) * true_sigma

print(f"Generated {n} observations with 3 predictors")
print(f"True coefficients: {true_beta}")
print(f"True sigma: {true_sigma}")

In [ ]:
# Purpose: Fit with NUTS (exact but slow)
# Key Concept: NUTS provides gold standard

with pm.Model() as regression_model:
    # Priors
    beta = pm.Normal('beta', mu=0, sigma=10, shape=3)
    sigma = pm.Exponential('sigma', 1)
    
    # Likelihood
    mu = pm.math.dot(X, beta)
    obs = pm.Normal('obs', mu=mu, sigma=sigma, observed=y)
    
    # NUTS sampling
    print("Sampling with NUTS...")
    start_nuts = time.time()
    trace_nuts = pm.sample(
        draws=1000,
        tune=1000,
        chains=2,
        random_seed=42,
        return_inferencedata=True,
        progressbar=False
    )
    time_nuts = time.time() - start_nuts

print(f"\nNUTS took {time_nuts:.2f} seconds")

In [ ]:
# Purpose: Fit with ADVI (approximate but fast)
# Key Concept: fit() method performs variational inference

with regression_model:
    print("Fitting with ADVI...")
    start_advi = time.time()
    
    # Perform variational inference
    approx = pm.fit(
        n=10000,  # Number of optimization iterations
        method='advi',
        random_seed=42,
        progressbar=False
    )
    
    # Sample from approximate posterior
    trace_advi = approx.sample(2000)
    
    time_advi = time.time() - start_advi

print(f"\nADVI took {time_advi:.2f} seconds")
print(f"Speedup: {time_nuts / time_advi:.1f}x faster")

# Final ELBO (higher is better)
print(f"\nFinal ELBO: {approx.hist[-1]:.2f}")

In [ ]:
# Purpose: Compare posterior estimates
# Key Concept: ADVI approximates NUTS well for simple models

# Extract summaries
summary_nuts = az.summary(trace_nuts, var_names=['beta', 'sigma'])
summary_advi = az.summary(trace_advi, var_names=['beta', 'sigma'])

print("\n" + "="*70)
print("COMPARISON: NUTS vs ADVI")
print("="*70)
print("\nNUTS:")
print(summary_nuts[['mean', 'sd']])
print("\nADVI:")
print(summary_advi[['mean', 'sd']])
print("\nTrue values:")
print(f"beta: {true_beta}")
print(f"sigma: {true_sigma}")

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

var_names = ['beta', 'sigma']
for i, var in enumerate(var_names):
    if var == 'beta':
        # Plot first coefficient
        nuts_samples = trace_nuts.posterior[var].values[:, :, 0].flatten()
        advi_samples = trace_advi.posterior[var].values[:, :, 0].flatten()
        true_val = true_beta[0]
        title = 'beta[0]'
    else:
        nuts_samples = trace_nuts.posterior[var].values.flatten()
        advi_samples = trace_advi.posterior[var].values.flatten()
        true_val = true_sigma
        title = 'sigma'
    
    axes[i*2].hist(nuts_samples, bins=50, alpha=0.6, label='NUTS', density=True)
    axes[i*2].axvline(true_val, color='r', linestyle='--', label='True')
    axes[i*2].set_title(f'{title} - NUTS')
    axes[i*2].legend()
    
    axes[i*2+1].hist(advi_samples, bins=50, alpha=0.6, label='ADVI', density=True, color='orange')
    axes[i*2+1].axvline(true_val, color='r', linestyle='--', label='True')
    axes[i*2+1].set_title(f'{title} - ADVI')
    axes[i*2+1].legend()

plt.suptitle('Posterior Comparison: NUTS vs ADVI', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

print("\n✅ ADVI approximates NUTS well for this simple model!")

## 4. ELBO Monitoring and Convergence

The ELBO trajectory tells us how well the optimization is progressing.

In [ ]:
# Purpose: Monitor ELBO during optimization
# Key Concept: ELBO should converge to a stable value

# Plot ELBO history
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(approx.hist)
plt.xlabel('Iteration')
plt.ylabel('ELBO')
plt.title('ELBO over Optimization')
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
# Zoom in on last 1000 iterations
plt.plot(approx.hist[-1000:])
plt.xlabel('Iteration (last 1000)')
plt.ylabel('ELBO')
plt.title('ELBO Convergence (zoomed)')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Check if converged
final_1000 = approx.hist[-1000:]
elbo_change = np.std(final_1000) / np.abs(np.mean(final_1000))
print(f"\nRelative ELBO std in last 1000 iterations: {elbo_change:.6f}")
if elbo_change < 0.01:
    print("✅ ELBO has converged")
else:
    print("⚠️ ELBO may not have converged - try more iterations")

## 5. Application: Large-Scale Commodity Model

VI shines when dealing with large datasets. Let's fit a hierarchical model to many commodities.

In [ ]:
# Purpose: Generate large-scale commodity dataset
# Key Concept: VI scales better than MCMC for big data

np.random.seed(456)
n_commodities = 20
n_days = 1000
n_predictors = 5

# Global parameters
global_beta_mean = np.array([1.0, -0.5, 0.3, 0.7, -0.2])
global_beta_std = 0.5

# Generate commodity-specific data
commodity_data = []
for i in range(n_commodities):
    # Commodity-specific coefficients
    beta_i = global_beta_mean + np.random.randn(n_predictors) * global_beta_std
    
    # Predictors
    X_i = np.random.randn(n_days, n_predictors)
    
    # Prices
    y_i = X_i @ beta_i + np.random.randn(n_days) * 0.5
    
    commodity_data.append({'X': X_i, 'y': y_i, 'beta': beta_i})

print(f"Generated data for {n_commodities} commodities")
print(f"Each has {n_days} days and {n_predictors} predictors")
print(f"Total observations: {n_commodities * n_days}")

In [ ]:
# Purpose: Build hierarchical model for all commodities
# Key Concept: Hierarchical structure pools information across commodities

# Stack data
X_all = np.vstack([d['X'] for d in commodity_data])
y_all = np.concatenate([d['y'] for d in commodity_data])
commodity_idx = np.repeat(np.arange(n_commodities), n_days)

print(f"Stacked data shape: X={X_all.shape}, y={y_all.shape}")

with pm.Model() as hierarchical_model:
    # Global hyperparameters
    mu_beta = pm.Normal('mu_beta', mu=0, sigma=5, shape=n_predictors)
    sigma_beta = pm.Exponential('sigma_beta', 1, shape=n_predictors)
    
    # Commodity-specific coefficients (non-centered)
    beta_raw = pm.Normal('beta_raw', mu=0, sigma=1, shape=(n_commodities, n_predictors))
    beta = pm.Deterministic(
        'beta',
        mu_beta[None, :] + beta_raw * sigma_beta[None, :]
    )
    
    # Observation noise
    sigma = pm.Exponential('sigma', 1)
    
    # Likelihood
    mu = pm.math.sum(X_all * beta[commodity_idx], axis=1)
    obs = pm.Normal('obs', mu=mu, sigma=sigma, observed=y_all)
    
    # Fit with ADVI
    print("\nFitting with ADVI (this may take a minute)...")
    start = time.time()
    
    approx_hier = pm.fit(
        n=20000,
        method='advi',
        random_seed=42,
        progressbar=True
    )
    
    time_elapsed = time.time() - start
    
    # Sample from posterior
    trace_hier = approx_hier.sample(2000)

print(f"\nADVI took {time_elapsed:.2f} seconds for {n_commodities * n_days} observations")
print(f"Final ELBO: {approx_hier.hist[-1]:.2f}")

In [ ]:
# Purpose: Analyze hierarchical posterior
# Key Concept: VI recovers global and commodity-specific parameters

# Extract global parameters
summary = az.summary(trace_hier, var_names=['mu_beta', 'sigma_beta', 'sigma'])

print("\n" + "="*60)
print("GLOBAL PARAMETERS")
print("="*60)
print(summary[['mean', 'sd']])
print(f"\nTrue global mean: {global_beta_mean}")
print(f"True global std: {global_beta_std}")

# Compare estimated vs true commodity-specific betas
beta_post_mean = trace_hier.posterior['beta'].mean(dim=['chain', 'draw']).values
beta_true = np.array([d['beta'] for d in commodity_data])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(beta_true.flatten(), beta_post_mean.flatten(), alpha=0.5)
axes[0].plot([beta_true.min(), beta_true.max()], 
             [beta_true.min(), beta_true.max()], 
             'r--', label='Perfect recovery')
axes[0].set_xlabel('True coefficient')
axes[0].set_ylabel('Estimated coefficient')
axes[0].set_title('Coefficient Recovery')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Error distribution
errors = (beta_post_mean - beta_true).flatten()
axes[1].hist(errors, bins=30, edgecolor='black')
axes[1].axvline(0, color='r', linestyle='--')
axes[1].set_xlabel('Estimation Error')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Error Distribution (RMSE={np.sqrt(np.mean(errors**2)):.3f})')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n✅ ADVI successfully recovered {n_commodities * n_predictors} parameters!")

## 6. Full-Rank vs Mean-Field ADVI

PyMC offers two ADVI variants:
- **Mean-field**: Assumes posterior independence (faster, less accurate)
- **Full-rank**: Models correlations (slower, more accurate)

In [ ]:
# Purpose: Compare mean-field and full-rank ADVI
# Key Concept: Full-rank captures posterior correlations

# Generate correlated data
np.random.seed(789)
n_obs = 200
rho_true = 0.7
theta_true = [2.0, 3.0]

# Correlated observations
cov_matrix = [[1.0, rho_true], [rho_true, 1.0]]
obs_data = np.random.multivariate_normal(theta_true, cov_matrix, size=n_obs)

with pm.Model() as corr_model:
    # Correlated prior
    theta = pm.MvNormal('theta', mu=[0, 0], cov=[[10, 0], [0, 10]], shape=2)
    
    # Likelihood
    obs = pm.MvNormal('obs', mu=theta, cov=cov_matrix, observed=obs_data)
    
    # Mean-field ADVI
    print("Fitting with mean-field ADVI...")
    approx_mf = pm.fit(n=10000, method='advi', random_seed=42, progressbar=False)
    trace_mf = approx_mf.sample(2000)
    
    # Full-rank ADVI
    print("Fitting with full-rank ADVI...")
    approx_fr = pm.fit(n=10000, method='fullrank_advi', random_seed=42, progressbar=False)
    trace_fr = approx_fr.sample(2000)
    
    # NUTS for comparison
    print("Sampling with NUTS...")
    trace_nuts_corr = pm.sample(1000, tune=1000, chains=2, random_seed=42, 
                                 return_inferencedata=True, progressbar=False)

# Compare correlations
def compute_corr(trace):
    samples = trace.posterior['theta'].values
    samples = samples.reshape(-1, 2)
    return np.corrcoef(samples.T)[0, 1]

corr_mf = compute_corr(trace_mf)
corr_fr = compute_corr(trace_fr)
corr_nuts = compute_corr(trace_nuts_corr)

print("\n" + "="*60)
print("POSTERIOR CORRELATION")
print("="*60)
print(f"True correlation: {rho_true:.3f}")
print(f"NUTS: {corr_nuts:.3f}")
print(f"Full-rank ADVI: {corr_fr:.3f}")
print(f"Mean-field ADVI: {corr_mf:.3f}")
print("\n⚠️ Mean-field assumes independence, underestimates correlation")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, trace, title in zip(axes, 
                             [trace_nuts_corr, trace_fr, trace_mf],
                             ['NUTS', 'Full-rank ADVI', 'Mean-field ADVI']):
    samples = trace.posterior['theta'].values.reshape(-1, 2)
    ax.scatter(samples[:, 0], samples[:, 1], alpha=0.3, s=10)
    ax.set_xlabel('theta[0]')
    ax.set_ylabel('theta[1]')
    ax.set_title(f'{title}\n(corr={np.corrcoef(samples.T)[0,1]:.3f})')
    ax.axvline(theta_true[0], color='r', linestyle='--', alpha=0.5)
    ax.axhline(theta_true[1], color='r', linestyle='--', alpha=0.5)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Exercises

### Exercise 6.4.1: ADVI Convergence Check

**Task:** Fit the model below with ADVI and determine if it has converged.

**Expected Output:** Convergence assessment and recommendation

In [ ]:
# Exercise setup
np.random.seed(999)
exercise_y = np.random.poisson(lam=5, size=100)

with pm.Model() as poisson_model:
    lam = pm.Gamma('lam', alpha=2, beta=0.5)
    obs = pm.Poisson('obs', mu=lam, observed=exercise_y)
    
    # Fit with ADVI
    approx_ex = pm.fit(n=5000, method='advi', random_seed=42, progressbar=False)

# YOUR CODE HERE
# ---------------
# 1. Plot ELBO history
# 2. Calculate ELBO change in last 1000 iterations
# 3. Determine if converged (relative std < 0.01)

has_converged = None  # True or False
recommendation = ""  # Your recommendation

# Your analysis code here

In [ ]:
# AUTO-GRADED TESTS
def test_exercise_6_4_1():
    assert has_converged is not None, "Set has_converged to True or False"
    assert isinstance(has_converged, bool), "has_converged should be boolean"
    assert recommendation != "", "Provide a recommendation"
    
    # Check ELBO analysis
    final_1000 = approx_ex.hist[-1000:]
    rel_std = np.std(final_1000) / np.abs(np.mean(final_1000))
    
    if rel_std < 0.01:
        assert has_converged == True, "ELBO has converged"
    else:
        assert has_converged == False, "ELBO has not converged"
    
    print("✅ Exercise 6.4.1 passed!")
    print(f"   Relative std: {rel_std:.6f}")
    print(f"   Your assessment: {'Converged' if has_converged else 'Not converged'}")

# Uncomment to test:
# test_exercise_6_4_1()

<details>
<summary>Solution</summary>

```python
# Plot ELBO
plt.figure(figsize=(12, 4))
plt.plot(approx_ex.hist)
plt.xlabel('Iteration')
plt.ylabel('ELBO')
plt.title('ELBO History')
plt.grid(alpha=0.3)
plt.show()

# Check convergence
final_1000 = approx_ex.hist[-1000:]
rel_std = np.std(final_1000) / np.abs(np.mean(final_1000))

has_converged = rel_std < 0.01

if has_converged:
    recommendation = "ELBO has converged. Results are reliable."
else:
    recommendation = "ELBO has not converged. Run more iterations (try n=10000)."

print(f"Relative std: {rel_std:.6f}")
print(recommendation)
```
</details>

### Exercise 6.4.2: Speed vs Accuracy Trade-off

**Task:** Compare ADVI and NUTS on computational cost and accuracy for a time series model.

In [ ]:
# Exercise setup: AR(1) model
np.random.seed(555)
n_t = 300
true_rho = 0.8
true_sigma = 0.5

ts_data = np.zeros(n_t)
ts_data[0] = np.random.randn()
for t in range(1, n_t):
    ts_data[t] = true_rho * ts_data[t-1] + np.random.randn() * true_sigma

# YOUR CODE HERE
# ---------------
# 1. Build AR(1) model
# 2. Fit with both NUTS and ADVI, recording time
# 3. Compare parameter estimates
# 4. Calculate relative error for each method

time_nuts_ex = None
time_advi_ex = None
error_nuts = None  # |estimated_rho - true_rho|
error_advi = None

# Your code here

In [ ]:
# AUTO-GRADED TESTS
def test_exercise_6_4_2():
    assert time_nuts_ex is not None, "Record NUTS time"
    assert time_advi_ex is not None, "Record ADVI time"
    assert error_nuts is not None, "Calculate NUTS error"
    assert error_advi is not None, "Calculate ADVI error"
    
    assert time_advi_ex < time_nuts_ex, "ADVI should be faster"
    
    speedup = time_nuts_ex / time_advi_ex
    
    print("✅ Exercise 6.4.2 passed!")
    print(f"   NUTS time: {time_nuts_ex:.2f}s")
    print(f"   ADVI time: {time_advi_ex:.2f}s")
    print(f"   Speedup: {speedup:.1f}x")
    print(f"   NUTS error: {error_nuts:.4f}")
    print(f"   ADVI error: {error_advi:.4f}")

# Uncomment to test:
# test_exercise_6_4_2()

### Exercise 6.4.3: When VI Fails

**Task:** Identify when ADVI produces poor approximations.

In [ ]:
# Exercise: Multimodal posterior
np.random.seed(666)

# Generate mixture data
n_mode1 = 50
n_mode2 = 50
mixture_data = np.concatenate([
    np.random.randn(n_mode1) - 3,
    np.random.randn(n_mode2) + 3
])

# YOUR CODE HERE
# ---------------
# 1. Fit Gaussian model with NUTS and ADVI
# 2. Compare posterior distributions
# 3. Explain why ADVI fails for multimodal posteriors

explanation = ""  # Your explanation

# Your code here

In [ ]:
# AUTO-GRADED TESTS
def test_exercise_6_4_3():
    assert explanation != "", "Provide explanation"
    assert 'multimodal' in explanation.lower() or 'mode' in explanation.lower(), \
        "Explanation should mention multimodality"
    
    print("✅ Exercise 6.4.3 passed!")
    print(f"\nYour explanation: {explanation}")

# Uncomment to test:
# test_exercise_6_4_3()

---

## Summary

### Key Takeaways

1. **VI is optimization**: Turns inference into finding best approximate distribution
2. **ELBO maximization**: Monitor ELBO convergence to check optimization
3. **Speed-accuracy trade-off**: ADVI is 10-100x faster but approximate
4. **Mean-field assumption**: Standard ADVI assumes posterior independence
5. **Full-rank option**: Captures correlations but slower
6. **Best for large data**: VI shines with thousands of observations
7. **Fails on multimodality**: VI struggles with complex posterior geometries
8. **Underestimates uncertainty**: Variational posteriors are often too confident

### When to Use VI

**Use ADVI when:**
- Dataset is large (>10,000 observations)
- Need fast iteration during model development
- Posterior is approximately unimodal
- Want deterministic results

**Use NUTS when:**
- Need exact posterior
- Model has complex geometry
- Posterior is multimodal
- Uncertainty quantification is critical

### What's Next

- **Next notebook**: Diagnosing Problems - comprehensive diagnostic workflow
- **Advanced topic**: Normalizing flows for better VI
- **Application**: Production forecasting with VI for real-time updates

### Additional Resources

- Blei et al. (2017): "Variational Inference: A Review for Statisticians"
- Kucukelbir et al. (2017): "Automatic Differentiation Variational Inference"
- PyMC VI tutorial: https://www.pymc.io/projects/docs/en/stable/learn/core_notebooks/variational_api_quickstart.html